# Random Forest Regressor Model Analysis
## Smart Queue Prediction System - IntelliQ

This notebook trains and evaluates the **Random Forest Regressor** model for predicting queue wait times.

**Metrics:** R2, Adjusted R2, MAE, MSE, RMSE, MAPE, Precision, Recall, F1-Score, Cross-Validation, Confusion Matrix

## 1. Upload Dataset
Upload `queue_management_refactored-general.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    explained_variance_score, max_error, median_absolute_error,
    classification_report, accuracy_score, confusion_matrix
)
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 3. Load & Explore Dataset

In [ ]:
df = pd.read_csv('queue_management_refactored-general.csv')
print('Dataset shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

## 4. Data Preprocessing

In [ ]:
categorical_columns = ['facility_id', 'service_type', 'priority_level', 'customer_type', 'queue_status']

cols_to_drop = ['customer_id', 'arrival_time']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

target_col = 'actual_wait_time'

if df[target_col].isnull().all():
    print('Warning: Target column is empty. Generating synthetic target data.')
    np.random.seed(42)
    base_wait = df['queue_length'].fillna(5) * df['avg_service_time'].fillna(10) / df['active_staff_count'].fillna(1).replace(0, 1)
    noise = np.random.normal(0, 5, size=len(df))
    df[target_col] = (base_wait + noise).clip(lower=0)
elif df[target_col].isnull().any():
    df = df.dropna(subset=[target_col]).copy()

y = df[target_col]
X = df.drop(columns=[target_col])

numeric_cols = X.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    median_val = X[col].median()
    if pd.isna(median_val): median_val = 0
    X[col] = X[col].fillna(median_val)

for col in categorical_columns:
    if col in X.columns:
        mode_series = X[col].mode()
        mode_val = mode_series.iloc[0] if not mode_series.empty else 'unknown'
        X[col] = X[col].fillna(mode_val)
        encoder = LabelEncoder()
        X[col] = encoder.fit_transform(X[col])

feature_names = list(X.columns)
print('Features:', feature_names)
print('X shape:', X.shape, '| y shape:', y.shape)

## 5. Train/Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]
n_samples = len(y_test)

print(f'Training set: {X_train_scaled.shape[0]} samples')
print(f'Test set:     {X_test_scaled.shape[0]} samples')
print(f'Features:     {n_features}')

## 6. Define Metrics Function

In [ ]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - n_features - 1)
    evs = explained_variance_score(y_true, y_pred)
    med_ae = median_absolute_error(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else float('nan')
    acc_10 = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) <= 0.10) * 100 if mask.sum() > 0 else float('nan')
    acc_20 = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) <= 0.20) * 100 if mask.sum() > 0 else float('nan')
    residuals = y_true - y_pred
    z_scores = np.abs(stats.zscore(residuals))
    return {
        'MAE': mae, 'MSE': mse, 'RMSE': rmse,
        'R2 Score': r2, 'Adjusted R2': adj_r2, 'Explained Variance': evs,
        'Median Abs Error': med_ae, 'Max Error': max_err, 'MAPE (%)': mape,
        'Accuracy (+/-10%)': acc_10, 'Accuracy (+/-20%)': acc_20,
        'Residual Mean': np.mean(residuals), 'Residual Std': np.std(residuals),
        'Outliers |Z|>2 (%)': np.mean(z_scores > 2) * 100
    }

## 7. Train Random Forest Regressor

In [ ]:
print('Training Random Forest Regressor...')
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

pred_train = model.predict(X_train_scaled)
pred_test = model.predict(X_test_scaled)
print('Random Forest Regressor trained successfully!')

## 8. Regression Metrics (Test Set)

In [ ]:
metrics = compute_metrics(y_test.values, pred_test)
metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
metrics_df['Value'] = metrics_df['Value'].apply(lambda x: f'{x:.4f}')
metrics_df.set_index('Metric')

## 9. Overfitting Check (Training vs Test)

In [ ]:
train_r2 = r2_score(y_train, pred_train)
test_r2 = metrics['R2 Score']
gap = train_r2 - test_r2

print(f'Training R2: {train_r2:.6f}')
print(f'Test R2:     {test_r2:.6f}')
print(f'Gap:         {gap:.6f}')

if gap < 0.05:
    print('\nNo significant overfitting detected.')
else:
    print('\nWarning: Potential overfitting detected!')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['Training R2', 'Test R2'], [train_r2, test_r2], color=['#4ECDC4', '#333333'])
ax.set_ylim(0, 1.1)
ax.set_title('Random Forest Regressor - Overfitting Check', fontweight='bold', fontsize=14)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. 5-Fold Cross-Validation

In [ ]:
X_all = np.vstack([X_train_scaled, X_test_scaled])
y_all = np.concatenate([y_train.values, y_test.values])

cv_scores = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42), X_all, y_all, cv=5, scoring='r2')

print('5-Fold Cross-Validation R2 Scores:')
for i, s in enumerate(cv_scores):
    print(f'  Fold {i+1}: {s:.6f}')
print(f'\n  Mean:    {cv_scores.mean():.6f}')
print(f'  Std Dev: {cv_scores.std():.6f}')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([f'Fold {i+1}' for i in range(5)], cv_scores, color='#4ECDC4', alpha=0.85, edgecolor='white')
ax.axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax.set_title('Random Forest Regressor - 5-Fold Cross-Validation R2', fontweight='bold', fontsize=14)
ax.set_ylabel('R2 Score')
ax.legend(fontsize=12)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{bar.get_height():.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 11. Classification Metrics (Crowd-Level Categories)
Wait times binned into: **Low** (0-10 min), **Medium** (10-25), **High** (25-50), **Critical** (50+)

In [ ]:
bins = [0, 10, 25, 50, float('inf')]
labels = ['Low', 'Medium', 'High', 'Critical']

y_true_cat = pd.cut(y_test, bins=bins, labels=labels, include_lowest=True)
y_pred_cat = pd.cut(pred_test, bins=bins, labels=labels, include_lowest=True).fillna('Critical')

acc = accuracy_score(y_true_cat, y_pred_cat)
print(f'Classification Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print()
print(classification_report(y_true_cat, y_pred_cat, zero_division=0))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_cat, y_pred_cat, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=ax,
            annot_kws={'size': 14})
ax.set_title('Random Forest Regressor - Confusion Matrix\nAccuracy: {acc*100:.1f}%', fontweight='bold', fontsize=14)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

## 12. Actual vs Predicted Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, pred_test, alpha=0.3, s=15, color='#4ECDC4', label='Predictions')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Wait Time (min)', fontsize=12)
ax.set_ylabel('Predicted Wait Time (min)', fontsize=12)
ax.set_title('Random Forest Regressor - Actual vs Predicted', fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 13. Residual Distribution

In [ ]:
residuals = y_test.values - pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=50, alpha=0.7, color='#4ECDC4', edgecolor='white')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Histogram\nMean: {np.mean(residuals):.2f} | Std: {np.std(residuals):.2f}', fontweight='bold')

# QQ Plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Normality Check)', fontweight='bold')

plt.suptitle('Random Forest Regressor - Residual Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. Feature Importance

In [ ]:
importances = model.feature_importances_
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(fi_df)))
ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors)
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest Regressor - Feature Importances', fontweight='bold', fontsize=14)
for i, (val, name) in enumerate(zip(fi_df['Importance'], fi_df['Feature'])):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 5 Features:')
fi_df.sort_values('Importance', ascending=False).head()

## 15. Summary Table (For PPT / SRS Report)

In [ ]:
report = classification_report(y_true_cat, y_pred_cat, zero_division=0, output_dict=True)

summary = {
    'Metric': [
        'R2 Score', 'Adjusted R2', 'MAE', 'RMSE', 'MAPE (%)',
        'Prediction Accuracy (+/-10%)', 'Prediction Accuracy (+/-20%)',
        'Classification Accuracy', 'Weighted F1-Score',
        'Cross-Val R2 (Mean +/- Std)',
        'Weighted Precision', 'Weighted Recall'
    ],
    'Random Forest Regressor': [
        f'{metrics["R2 Score"]:.4f}',
        f'{metrics["Adjusted R2"]:.4f}',
        f'{metrics["MAE"]:.4f}',
        f'{metrics["RMSE"]:.4f}',
        f'{metrics["MAPE (%)"]:.2f}%',
        f'{metrics["Accuracy (+/-10%)"]:.2f}%',
        f'{metrics["Accuracy (+/-20%)"]:.2f}%',
        f'{acc*100:.2f}%',
        f'{report["weighted avg"]["f1-score"]:.4f}',
        f'{cv_scores.mean():.4f} +/- {cv_scores.std():.4f}',
        f'{report["weighted avg"]["precision"]:.4f}',
        f'{report["weighted avg"]["recall"]:.4f}'
    ]
}

summary_df = pd.DataFrame(summary).set_index('Metric')
summary_df.style.set_caption('Random Forest Regressor - Performance Summary')

## Conclusion

The **Random Forest Regressor** model has been trained and evaluated on the IntelliQ Smart Queue dataset.
All metrics above can be directly used in the SRS Report and PPT documentation.